In [0]:
bronze_path = "abfss://bronze@storageaccountspotifyuk.dfs.core.windows.net/hospitals"
silver_path = "abfss://silver@storageaccountspotifyuk.dfs.core.windows.net/hospitals"
required_columns = ['facility_id',
    'facility_name',
    'address',
    'city',
    'state',
    'zip_code',
    'county',
    'telephone_number',
    'hospital_type',
    'hospital_ownership',
    'emergency_services',
    'meets_criteria_for_birthing_friendly_designation',
    'hospital_overall_rating',
    'ingestion_ts',
    'source_file']

In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.format("delta").option("header", True).option("inferSchema", True).load(bronze_path).select(*required_columns)

In [0]:
display(df.limit(10))

In [0]:
df = df.withColumnsRenamed({
    "meets_criteria_for_birthing_friendly_designation": "meets_criteria_for_maternity",
    "hospital_overall_rating": "rating",
})

In [0]:
df = df.withColumn("rating", when(col("rating").startswith("Not"), 0).otherwise(col("rating")))

In [0]:
from pyspark.sql.functions import col, when, upper, expr
spark.conf.set("spark.sql.ansi.enabled", "false")

df = df.withColumns({
    "class": when(col("rating") > 3, "High")
             .when(col("rating") > 2, "Medium")
             .otherwise("Low"),
    # Use expr with try_cast to tolerate malformed input
    "facility_id": expr("try_cast(facility_id as int)"),
    # upper case
    "facility_name": upper(col("facility_name")),    "emergency_services": when(col("emergency_services") == True, "Y").when(col("emergency_services") == False, "N").otherwise("Unknown"),

    "address": upper(col("address")),
    "city": upper(col("city")),
    "state": upper(col("state")),

    # boolean
    "has_emergency_service": col("emergency_services").contains("Y"),
})
display(df)

In [0]:
# delete duplicate records but keep first row... going to use window functions
# df = df.dropDuplicates(['facility_id'])

from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

w = Window.partitionBy("facility_id").orderBy(col("facility_id").asc())

df = df.withColumn("row_num", row_number().over(w))
df = df.filter(col("row_num") == 1).drop("row_num")
display(df.limit(10))

## Path vs Table — When to Use What
### Use path-based tables when:
- No Unity Catalog
- Simple learning projects
- Raw / staging zones
- External pipelines

### Use Unity Catalog tables when:
- Enterprise projects
- Security & governance needed
- BI users / analysts involved

In [0]:
from delta.tables import DeltaTable
do_it_when_uc_is_not_enabled = False
if do_it_when_uc_is_not_enabled:
    silver_table = DeltaTable.forPath(spark, silver_path)
    silver_table.alias("o")\
    .merge(df.alias("n"), "o.facility_id = n.facility_id")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

In [0]:
table_name = "healthcare_catalog.silver.hospitals"
if not spark.catalog.tableExists(table_name):
    df.limit(0) \
        .write \
        .format("delta") \
        .saveAsTable(table_name)

In [0]:
# incremental loading data in silver table (Recommended if UC is enabled)
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(spark, table_name)

silver_table.alias("o")\
    .merge(df.alias("n"), "o.facility_id = n.facility_id")\
    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()